In [8]:
# ==========================================================
# 1️⃣ IMPORTS
# ==========================================================
import pandas as pd
import numpy as np
import ast
import json
from scipy.stats import entropy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================================
# 2️⃣ LOAD DATA
# ==========================================================
df = pd.read_csv("iiot_dataset_clean.csv")

In [9]:
df.head(100)

,device_name_enc,application_enc,ip_enc,topic_enc,message_type_enc,qos,retained,duplicate,message_val_num,hour,minute,second,label
0,11,10,7,20,1,0,0,0,280.0,23,35,57,0
1,1,7,2,16,1,0,0,0,32.0,23,35,58,0
2,5,12,1,22,1,0,0,0,1023.0,23,35,58,0
3,0,13,0,29,1,0,0,0,23.2,23,35,58,0
4,0,13,0,28,1,0,0,0,42.0,23,35,58,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0,13,0,24,1,0,0,0,-50.5,23,36,10,0
96,13,3,8,6,1,0,0,0,0.0,23,36,10,0
97,13,3,8,8,0,0,0,0,0.0,23,36,10,0
98,8,6,12,15,1,0,0,0,0.0,23,36,10,0


In [10]:
print("Shape :", df.shape)
df.info()

Shape : (1115684, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115684 entries, 0 to 1115683
Data columns (total 13 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   device_name_enc   1115684 non-null  int64  
 1   application_enc   1115684 non-null  int64  
 2   ip_enc            1115684 non-null  int64  
 3   topic_enc         1115684 non-null  int64  
 4   message_type_enc  1115684 non-null  int64  
 5   qos               1115684 non-null  int64  
 6   retained          1115684 non-null  int64  
 7   duplicate         1115684 non-null  int64  
 8   message_val_num   1115684 non-null  float64
 9   hour              1115684 non-null  int64  
 10  minute            1115684 non-null  int64  
 11  second            1115684 non-null  int64  
 12  label             1115684 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 110.7 MB


Colonnes constantes (inutiles)

Ces colonnes ont une seule valeur (0) sur tout le dataset :

qos

retained

duplicate

➡ Elles n'apportent aucune information au modèle.
➡ Elles doivent être supprimées.

In [11]:
df = df.drop(columns=["qos", "retained", "duplicate"])

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115684 entries, 0 to 1115683
Data columns (total 10 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   device_name_enc   1115684 non-null  int64  
 1   application_enc   1115684 non-null  int64  
 2   ip_enc            1115684 non-null  int64  
 3   topic_enc         1115684 non-null  int64  
 4   message_type_enc  1115684 non-null  int64  
 5   message_val_num   1115684 non-null  float64
 6   hour              1115684 non-null  int64  
 7   minute            1115684 non-null  int64  
 8   second            1115684 non-null  int64  
 9   label             1115684 non-null  int64  
dtypes: float64(1), int64(9)
memory usage: 85.1 MB


Problème détecté : Outliers importants
message_val_num

Q1 = 13

Q3 = 253

IQR = 240

Seuil supérieur IQR = 613

Nombre d’outliers = 151 445 lignes (~13.5%)

Valeur max = 102 138
Valeur min = -67.72

👉 Distribution très asymétrique (fortement skewed).
👉 Cela peut fortement perturber les modèles ML.

Gérer les outliers

🔵 Option recommandée : Transformation logarithmique

Comme les valeurs sont très dispersées :

Pourquoi ?

Réduit l’effet des valeurs extrêmes

Stabilise la variance

Améliore les performances des modèles

In [13]:
import numpy as np

df["message_val_num"] = np.log1p(df["message_val_num"].clip(lower=0))

Vérifier les valeurs négatives

Si les valeurs négatives ne sont pas logiquement possibles :

In [14]:
df = df[df["message_val_num"] >= 0]

Tu peux aussi :

Transformer hour en variable cyclique :

In [15]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

In [17]:
df.head(100)

,device_name_enc,application_enc,ip_enc,topic_enc,message_type_enc,message_val_num,hour,minute,second,label,hour_sin,hour_cos
0,11,10,7,20,1,5.638355,23,35,57,0,-0.258819,0.965926
1,1,7,2,16,1,3.496508,23,35,58,0,-0.258819,0.965926
2,5,12,1,22,1,6.931472,23,35,58,0,-0.258819,0.965926
3,0,13,0,29,1,3.186353,23,35,58,0,-0.258819,0.965926
4,0,13,0,28,1,3.761200,23,35,58,0,-0.258819,0.965926
...,...,...,...,...,...,...,...,...,...,...,...,...
95,0,13,0,24,1,0.000000,23,36,10,0,-0.258819,0.965926
96,13,3,8,6,1,0.000000,23,36,10,0,-0.258819,0.965926
97,13,3,8,8,0,0.000000,23,36,10,0,-0.258819,0.965926
98,8,6,12,15,1,0.000000,23,36,10,0,-0.258819,0.965926


In [18]:
print(df["label"].value_counts())

label
0    854338
1    261346
Name: count, dtype: int64


In [23]:
df['label'] = df['label'].astype(int)

In [26]:
# 2. Suppression des colonnes demandées
# On retire hour_sin et hour_cos comme convenu
cols_to_remove = ['hour_sin', 'hour_cos','minute','second']
df = df.drop(columns=[c for c in cols_to_remove if c in df.columns])

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115684 entries, 0 to 1115683
Data columns (total 7 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   device_name_enc   1115684 non-null  int64  
 1   application_enc   1115684 non-null  int64  
 2   ip_enc            1115684 non-null  int64  
 3   topic_enc         1115684 non-null  int64  
 4   message_type_enc  1115684 non-null  int64  
 5   message_val_num   1115684 non-null  float64
 6   label             1115684 non-null  int64  
dtypes: float64(1), int64(6)
memory usage: 59.6 MB


In [28]:
# 1. Sauvegarde de sécurité
df_processed = df.copy()
df_processed.to_csv('data_clean_recon.csv', index=False)